# Colab 03 · Fine-tune transformers for review sentiment (DistilBERT + RoBERTa)

**Goal:** fine-tune two transformers on our 31.5k training reviews and score them exactly like the local models.
- **DistilBERT** (`distilbert-base-uncased`): a general language model with a new 3-class head.
- **RoBERTa** (`cardiffnlp/twitter-roberta-base-sentiment-latest`): already a sentiment model (trained on tweets). We tested it *as it is* locally; here we **improve it on our review data**, so the slides can show before vs after.

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU). About 25–35 minutes in total.

**Inputs (on your Google Drive):** `MyDrive/rrf/data/processed/text_{train,val,test,external_test}.csv`

**Outputs:** metrics, predictions and confusion matrices are copied to `MyDrive/rrf/results/`; the fine-tuned models are saved to `MyDrive/rrf/models/` (too big for git).

In [ ]:
# ---- Setup: GPU check, Google Drive, code, packages ----
import os, subprocess, sys
from pathlib import Path

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
assert gpu, "No GPU! Menu: Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", gpu)

from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/rrf")          # project folder on your Drive
REPO = Path("/content/application-of-text-qualification")

# The GitHub repo is private, so cloning needs a token stored in Colab "Secrets" (key icon
# in the left sidebar) under the name GITHUB_TOKEN, with notebook access switched on.
if not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q",
                    f"https://{token}@github.com/Flaxxxy/application-of-text-qualification.git", str(REPO)],
                   check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("ready:", REPO)

## 1. Get the processed data from Drive

In [ ]:
import shutil
src_dir, dst_dir = DRIVE / "data" / "processed", REPO / "data" / "processed"
dst_dir.mkdir(parents=True, exist_ok=True)
for name in ["train", "val", "test", "external_test"]:
    shutil.copy(src_dir / f"text_{name}.csv", dst_dir)

import pandas as pd
data = {s: pd.read_csv(dst_dir / f"text_{s}.csv") for s in ["train", "val", "test", "external_test"]}
{k: len(v) for k, v in data.items()}

## 2. Fine-tune and evaluate both models

The training code is `finetune_transformer()` in `src/text_models.py`; scoring uses the same
`run_and_score()` as every local model, so the numbers are directly comparable.
Speed is not reported here, because a GPU is not comparable to the CPU timings of the local models.

In [ ]:
import json
from src.config import load_config, path, set_seed
from src.evaluate import plot_confusion_matrix
from src.text_models import FineTunedModel, finetune_transformer, run_and_score, save_predictions

set_seed()
ft = load_config()["text"]["finetune"]
results = {}
for name, base_model in ft["models"].items():
    print(f"\n===== {name}  ({base_model}) =====")
    out_dir = DRIVE / "models" / f"text_{name}"
    trainer = finetune_transformer(base_model, data["train"], data["val"], f"/content/tmp_{name}",
                                   epochs=ft["epochs"], max_length=ft["max_length"],
                                   batch_size=ft["batch_size"], lr=ft["learning_rate"])
    trainer.save_model(str(out_dir))          # best epoch (by validation macro-F1) + tokenizer
    model = FineTunedModel(name, out_dir, max_length=ft["max_length"])
    metrics, proba = run_and_score(model, data["test"], data["external_test"])
    metrics.update({"ms_per_review": None, "base_model": base_model,
                    "chosen_on_val": {"epochs": ft["epochs"], "lr": ft["learning_rate"],
                                      "best_val_macro_f1": round(trainer.state.best_metric, 4)}})
    path("results", "colab", create=True).joinpath(f"metrics_text_{name}.json").write_text(json.dumps(metrics, indent=2))
    save_predictions(name, data["test"], proba)
    plot_confusion_matrix(metrics["test"]["confusion_matrix"], metrics["test"]["labels"],
                          f"{name}  (macro-F1 {metrics['test']['macro_f1']:.3f})", f"cm_text_{name}")
    results[name] = metrics
    print(f"{name}: test macro-F1 {metrics['test']['macro_f1']:.4f} | external F1 {metrics['external']['macro_f1']:.4f}")

In [ ]:
pd.DataFrame({n: {"test macro-F1": m["test"]["macro_f1"], "test accuracy": m["test"]["accuracy"],
                  "external F1 (pos/neg)": m["external"]["macro_f1"]} for n, m in results.items()}).T

## 3. Copy results to Drive and list what to bring back

In [ ]:
files = []
for name in results:
    files += [f"results/colab/metrics_text_{name}.json",
              f"results/predictions/text_{name}_test.csv",
              f"results/figures/cm_text_{name}.png"]
for f in files:
    dst = DRIVE / f
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(REPO / f, dst)

print("Copy these files from Google Drive (MyDrive/rrf/...) into the SAME paths in the repo:\n")
for f in files:
    print("  ", f)
print("\nThe fine-tuned models stay on Drive in MyDrive/rrf/models/ (not needed in the repo).")

## Key findings

*(Filled in after the run, from the table above: which model is best, how much fine-tuning
improved RoBERTa compared to using it as it is, and where the models still make mistakes.)*